## Project setup and dependencies

In [ ]:
!pip install langchain-fireworks langchain-openai langchain-community \
            langchain-qdrant ragas langsmith langgraph \
            sentence-transformers qdrant-client pypdf

## Imports

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_fireworks import ChatFireworks
from langchain_fireworks import FireworksEmbeddings
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, List
import langsmith

/Users/michaeldoran/AIE9/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Fireworks Endpoints

In [5]:
# Your deployed endpoints from Fireworks dashboard
FIREWORKS_LLM_ENDPOINT    = "accounts/fireworks/models/gpt-oss-20b"
FIREWORKS_EMBED_ENDPOINT  = "accounts/fireworks/models/qwen3-embedding-8b"

# Fireworks LLM
fw_llm = ChatFireworks(
    model=FIREWORKS_LLM_ENDPOINT,
    temperature=0,
    max_tokens=1024,
)

# Fireworks Embeddings
fw_embeddings = FireworksEmbeddings(
    model=FIREWORKS_EMBED_ENDPOINT,
)

# OpenAI equivalents for comparison
oai_llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,
    max_tokens=1024,
)

oai_embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
)

Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x127380800>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x1273822d0>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x125f2b1d0>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x126b7b230>


## Corpus and Vector Store

In [6]:
# Sample corpus 
docs = [
    Document(page_content="LangGraph is a framework for building stateful, multi-actor LLM applications using graph-based workflows."),
    Document(page_content="LangSmith provides tracing, evaluation, and monitoring for LLM applications built with LangChain."),
    Document(page_content="Retrieval Augmented Generation (RAG) combines vector search with LLM generation to answer questions grounded in source documents."),
    Document(page_content="Fireworks AI provides fast, cost-efficient inference for open-source models including Llama, Mixtral, and Qwen."),
    Document(page_content="RAGAS is an evaluation framework for RAG pipelines measuring faithfulness, answer relevancy, and context precision."),
    Document(page_content="The MCP protocol standardizes how AI agents connect to external tools and data sources over HTTP."),
    Document(page_content="Vector stores index document embeddings and support similarity search to retrieve relevant context for LLM queries."),
    Document(page_content="LangChain tools allow agents to call external APIs, search the web, query databases, and run code."),
]

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
splits = splitter.split_documents(docs)

# Build two separate vector stores — one per embedding provider
fw_vectorstore  = InMemoryVectorStore.from_documents(splits, fw_embeddings)
oai_vectorstore = InMemoryVectorStore.from_documents(splits, oai_embeddings)

fw_retriever  = fw_vectorstore.as_retriever(search_kwargs={"k": 3})
oai_retriever = oai_vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"Indexed {len(splits)} chunks into both vector stores")

Indexed 8 chunks into both vector stores


### Rag Graph!

In [7]:
class RAGState(TypedDict):
    question: str
    contexts: List[str]
    answer: str

RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer the question using only the provided context. "
               "If the context doesn't contain the answer, say 'I don't know'."),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])

def build_rag_graph(llm, retriever, run_name: str):
    """Build a LangGraph RAG pipeline for a given LLM + retriever pair."""

    def retrieve_node(state: RAGState) -> dict:
        docs = retriever.invoke(state["question"])
        return {"contexts": [d.page_content for d in docs]}

    def generate_node(state: RAGState) -> dict:
        context = "\n\n".join(state["contexts"])
        chain = RAG_PROMPT | llm
        response = chain.invoke({
            "context": context,
            "question": state["question"],
        })
        return {"answer": response.content}

    graph = (
        StateGraph(RAGState)
        .add_node("retrieve", retrieve_node)
        .add_node("generate", generate_node)
        .add_edge(START, "retrieve")
        .add_edge("retrieve", "generate")
        .add_edge("generate", END)
        .compile()
    )
    return graph

# Build both pipelines
fw_rag  = build_rag_graph(fw_llm,  fw_retriever,  "fireworks-rag")
oai_rag = build_rag_graph(oai_llm, oai_retriever, "openai-rag")

## Run This

In [11]:
import time
from fireworks.client.error import RateLimitError as FireworksRateLimitError
from openai import RateLimitError as OpenAIRateLimitError

def run_with_backoff(fn, question, max_retries=5):
    delay = 5
    for attempt in range(max_retries):
        try:
            return fn(question)
        except (FireworksRateLimitError, OpenAIRateLimitError):
            if attempt == max_retries - 1:
                raise
            print(f"Rate limited. Waiting {delay}s before retry (attempt {attempt + 1}/{max_retries})...")
            time.sleep(delay)
            delay *= 2

fw_results = []
for q in test_questions:
    print(f"Running Fireworks: {q[:50]}...")
    fw_results.append(run_with_backoff(run_fw_rag, q))
    time.sleep(3)

oai_results = []
for q in test_questions:
    oai_results.append(run_oai_rag(q))
    time.sleep(1)

print("Both pipelines completed.")
for i, q in enumerate(test_questions):
    print(f"\nQ: {q}")
    print(f"  Fireworks: {fw_results[i]['answer'][:100]}...")
    print(f"  OpenAI:    {oai_results[i]['answer'][:100]}...")

Running Fireworks: What is LangGraph used for?...
Running Fireworks: How does RAG work?...
Running Fireworks: What does RAGAS measure?...
Running Fireworks: What is Fireworks AI?...
Running Fireworks: What is LangSmith used for?...
Running Fireworks: What is the MCP protocol?...
Rate limited. Waiting 5s before retry (attempt 1/5)...
Rate limited. Waiting 10s before retry (attempt 2/5)...
Running Fireworks: How do vector stores work?...
Rate limited. Waiting 5s before retry (attempt 1/5)...
Rate limited. Waiting 10s before retry (attempt 2/5)...
Both pipelines completed.

Q: What is LangGraph used for?
  Fireworks: LangGraph is a framework for building stateful, multi‑actor LLM applications that use graph‑based wo...
  OpenAI:    LangGraph is used for building stateful, multi-actor LLM applications using graph-based workflows....

Q: How does RAG work?
  Fireworks: **Retrieval‑Augmented Generation (RAG) works in two main stages:**

1. **Retrieval** –  
   * The us...
  OpenAI:    RAG (R

## Ragas Time!

In [13]:
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from datasets import Dataset
from ragas.metrics import faithfulness, answer_relevancy, context_precision
metrics = [faithfulness, answer_relevancy, context_precision]

reference_answers = {
    "What is LangGraph used for?": "LangGraph is used for building stateful, multi-actor LLM applications using graph-based workflows.",
    "How does RAG work?": "RAG works by splitting documents into chunks, embedding them into a vector store, retrieving relevant chunks via similarity search, and using them as context for LLM generation.",
    "What does RAGAS measure?": "RAGAS is an evaluation framework for RAG pipelines that measures faithfulness, answer relevancy, and context precision.",
    "What is Fireworks AI?": "Fireworks AI provides fast, cost-efficient inference for open-source models including Llama, Mixtral, and Qwen.",
    "What is LangSmith used for?": "LangSmith is used for tracing, evaluation, and monitoring of LLM applications built with LangChain.",
    "What is the MCP protocol?": "The MCP protocol standardizes how AI agents connect to external tools and data sources over HTTP.",
    "How do vector stores work?": "Vector stores index document embeddings and support similarity search to retrieve relevant context for LLM queries.",
}

def build_ragas_dataset(questions, results, retriever):
    rows = []
    for q, r in zip(questions, results):
        retrieved = retriever.invoke(q)
        rows.append({
            "question":  q,
            "answer":    r["answer"],
            "contexts":  [d.page_content for d in retrieved],
            "reference": reference_answers[q],
        })
    return Dataset.from_list(rows)

# Build datasets
fw_dataset  = build_ragas_dataset(test_questions, fw_results,  fw_retriever)
oai_dataset = build_ragas_dataset(test_questions, oai_results, oai_retriever)

# Wrap LLMs and embeddings for RAGAS
# Use OpenAI as the judge LLM for both evaluations (fair comparison)
judge_llm   = LangchainLLMWrapper(oai_llm)
judge_embed = LangchainEmbeddingsWrapper(oai_embeddings)

metrics = [faithfulness, answer_relevancy, context_precision]

print("Evaluating Fireworks pipeline...")
fw_eval = evaluate(
    fw_dataset,
    metrics=metrics,
    llm=judge_llm,
    embeddings=judge_embed,
)

print("Evaluating OpenAI pipeline...")
oai_eval = evaluate(
    oai_dataset,
    metrics=metrics,
    llm=judge_llm,
    embeddings=judge_embed,
)

/var/folders/kt/kthn0r7n3tj6bc8j8w5_glkh0000gn/T/ipykernel_760/2600581947.py:5: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision
/var/folders/kt/kthn0r7n3tj6bc8j8w5_glkh0000gn/T/ipykernel_760/2600581947.py:5: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_precision
/var/folders/kt/kthn0r7n3tj6bc8j8w5_glkh0000gn/T/ipykernel_760/2600581947.py:5: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Exampl

Evaluating Fireworks pipeline...


Evaluating:  10%|▉         | 2/21 [00:03<00:29,  1.57s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating: 100%|██████████| 21/21 [00:28<00:00,  1.36s/it]


Evaluating OpenAI pipeline...


Evaluating:  10%|▉         | 2/21 [00:04<00:38,  2.05s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating: 100%|██████████| 21/21 [00:21<00:00,  1.02s/it]


## Results!

In [14]:
import pandas as pd

fw_df  = fw_eval.to_pandas()
oai_df = oai_eval.to_pandas()

summary = pd.DataFrame({
    "Metric": ["faithfulness", "answer_relevancy", "context_precision"],
    "Fireworks (gpt-oss-20b)": [
        fw_df["faithfulness"].mean(),
        fw_df["answer_relevancy"].mean(),
        fw_df["context_precision"].mean(),
    ],
    "OpenAI (gpt-4.1-mini)": [
        oai_df["faithfulness"].mean(),
        oai_df["answer_relevancy"].mean(),
        oai_df["context_precision"].mean(),
    ],
})

print("\n=== RAGAS Evaluation Summary ===")
print(summary.to_string(index=False))


=== RAGAS Evaluation Summary ===
           Metric  Fireworks (gpt-oss-20b)  OpenAI (gpt-4.1-mini)
     faithfulness                 0.731519               1.000000
 answer_relevancy                 0.844370               0.864818
context_precision                 0.857143               0.928571


## Cost Comparison

In [ ]:
from langsmith import Client

ls_client = Client()

def get_cost_for_tag(tag: str) -> dict:
    """Pull token usage and cost from LangSmith traces by tag."""
    runs = list(ls_client.list_runs(
        project_name=os.environ["LANGCHAIN_PROJECT"],
        filter=f'has(tags, "{tag}")',
        run_type="chain",
    ))
    total_input  = sum(r.prompt_tokens    or 0 for r in runs)
    total_output = sum(r.completion_tokens or 0 for r in runs)
    total_cost   = sum(r.total_cost        or 0 for r in runs)
    return {
    "tag":            tag,
    "runs":           len(runs),
    "input_tokens":   total_input,
    "output_tokens":  total_output,
    "total_cost_usd": round(float(total_cost), 6),
}

fw_cost  = get_cost_for_tag("fireworks")
oai_cost = get_cost_for_tag("openai")

cost_df = pd.DataFrame([fw_cost, oai_cost])
print("\n=== Cost Comparison (LangSmith) ===")
print(cost_df.to_string(index=False))
print(f"\nCost ratio (OpenAI / Fireworks): {float(oai_cost['total_cost_usd']) / max(float(fw_cost['total_cost_usd']), 0.000001):.2f}x")

## Conclusion: 



In [17]:
conclusion = """
=== Evaluation Conclusion ===

We compared two RAG pipelines — Fireworks AI (gpt-oss-20b + qwen3-embedding-8b)
against OpenAI (gpt-4.1-mini + text-embedding-3-small) — across quality and cost dimensions.

--- Quality (RAGAS) ---

| Metric            | Fireworks (gpt-oss-20b) | OpenAI (gpt-4.1-mini) |
|-------------------|-------------------------|-----------------------|
| Faithfulness      | 0.73                    | 1.00                  |
| Answer Relevancy  | 0.84                    | 0.86                  |
| Context Precision | 0.86                    | 0.93                  |

The most significant gap is faithfulness (0.73 vs 1.0): gpt-oss-20b occasionally
generates content not strictly grounded in retrieved context. Answer relevancy
and context precision gaps are smaller — both models answer the question asked
and retrieve reasonably ranked context.

--- Cost (LangSmith) ---

| Provider  | Runs | Input Tokens | Output Tokens | Cost (USD) |
|-----------|------|--------------|---------------|------------|
| Fireworks | 141  | 12,904       | 14,412        | ~$0.00     |
| OpenAI    | 35   | 3,192        | 600           | $0.002237  |

Cost ratio: ~2,237x in favor of Fireworks at this scale.

--- Verdict ---

For high-volume RAG applications where cost is the primary constraint and
occasional hallucination is acceptable, Fireworks AI offers a compelling
open-source alternative. For use cases requiring strict factual grounding
(legal, medical, compliance), OpenAI's perfect faithfulness score justifies
the cost premium. The answer relevancy parity suggests gpt-oss-20b understands
the task — it just doesn't always stay within the retrieved context.
"""

print(conclusion)


=== Evaluation Conclusion ===

We compared two RAG pipelines — Fireworks AI (gpt-oss-20b + qwen3-embedding-8b)
against OpenAI (gpt-4.1-mini + text-embedding-3-small) — across quality and cost dimensions.

--- Quality (RAGAS) ---

| Metric            | Fireworks (gpt-oss-20b) | OpenAI (gpt-4.1-mini) |
|-------------------|-------------------------|-----------------------|
| Faithfulness      | 0.73                    | 1.00                  |
| Answer Relevancy  | 0.84                    | 0.86                  |
| Context Precision | 0.86                    | 0.93                  |

The most significant gap is faithfulness (0.73 vs 1.0): gpt-oss-20b occasionally
generates content not strictly grounded in retrieved context. Answer relevancy
and context precision gaps are smaller — both models answer the question asked
and retrieve reasonably ranked context.

--- Cost (LangSmith) ---

| Provider  | Runs | Input Tokens | Output Tokens | Cost (USD) |
|-----------|------|--------------|--